# Qwen3-ASR 会議文字起こし

環境: **GPU = T4** / Colab Python ランタイム **2026.4**

<!-- 詳細（この行と末尾のコメント記号を外すと下の表が表示される）

## 推奨環境
| 項目 | 指定 | 備考 |
|---|---|---|
| ランタイム | **GPU** | ノートのメタデータで指定済み（開くと自動でGPUランタイムを選択） |
| GPU | T4 / L4 / A100 いずれも可 | `cc>=8`(Ampere以降)は flash-attn + bf16、T4等(cc<8)は sdpa + fp16 に分岐 |
| Python | 3.12（Colab既定） | Colabはランタイム画像でPython固定。ノート側からは変更不可 |
| 主要パッケージ | `qwen-asr` / `torch 2.11 (cu128)` / `transformers 4.57` / `flash_attn 2.8.3`(cc>=8のみ) | pip installセルで導入 |

> - GPUの種類は自動判定されるので、どのランタイムでもセルを触らずそのまま実行可。
> - 無料Colabは基本 T4。L4 / A100 は Colab Pro が必要。
> - デフォルトGPUを変えたい場合はメニュー「ランタイム → ランタイムのタイプを変更」で選択。

-->


In [ ]:
#@title pip install（qwen3-asrとpyannote）

#@markdown # ⚠️要再起動
#@markdown `pyannote-metrics 4.1 depends on numpy>=2.2.2`
#@markdown
#@markdown `numba 0.60.0 depends on numpy<2.1 and >=1.22`
#@markdown
#@markdown ※pyannoteの上記のせいでNumba, Numpyの順に確実にバージョンアップするので**再起動必須！**
#@markdown
#@markdown ※再起動後はpip install終わった状態になるので**次のセルから実行で問題なし**
#@markdown
#@markdown ---
#@markdown **BUILD_FLASH_IF_MISSING**：flash-attn の whl （[lesj0610氏](https://github.com/lesj0610/flash-attention]https://github.com/lesj0610/flash-attention)）が消えた場合にソースからビルド（20〜30分）
#@markdown
#@markdown （Falseならビルドせずsdpaで続行）
BUILD_FLASH_IF_MISSING = False  #@param {type:"boolean"}


# flash-attention は Ampere(cc>=8)以降だけ導入（T4はスキップ）
#   ① 配布済みwhl(高速・数十秒)を優先。ただし個人フォークのGitHubリリース依存なので、
#      Private化/削除されると404で取得不可。
#   ② その場合、上のチェックボックスがONならソースビルド、OFF(既定)ならスキップ。
#   ③ 入らなければモデル準備セルが自動で sdpa に切り替わる(通知あり・動作は継続)。
import os, sys, subprocess, torch

FLASH_WHL = "https://github.com/lesj0610/flash-attention/releases/download/v2.8.3-cu12-torch2.11/flash_attn-2.8.3%2Bcu12torch2.11cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"

def _flash_ok():
    try:
        import flash_attn  # noqa: F401
        return True
    except Exception:
        return False

if torch.cuda.get_device_capability(0)[0] < 8:
    print(f"ℹ️ GPU {torch.cuda.get_device_name(0)} (cc<8): flash-attn不要のためスキップ")
elif _flash_ok():
    print("✅ flash-attn は既にインストール済み")
else:
    print("配布済みwhlを取得中...")
    subprocess.run([sys.executable, "-m", "pip", "install", FLASH_WHL])
    if not _flash_ok():
        # whl取得失敗(削除/Private化など)
        if BUILD_FLASH_IF_MISSING:
            print("whl取得失敗 → チェックONのためソースからビルド")
            subprocess.run([sys.executable, "-m", "pip", "install", "ninja"])  # ビルド高速化
            subprocess.run([sys.executable, "-m", "pip", "install", "flash-attn==2.8.3", "--no-build-isolation"],
                           env=dict(os.environ, MAX_JOBS="4"))
        else:
            print("whlを取得不可（削除/Private化の可能性）。"
                  "ビルドはスキップします")
    # 最終判定
    if _flash_ok():
        print("flash-attn 利用可能")
    else:
        print("flash-attn無しで続行")

# ※ qwen-asr は 0.0.x 系でAPI変更が入りやすいため、動作確認済みバージョンにピン留め。
#   最新を試したいときだけ ==0.0.6 を外す(その場合 from_pretrained/transcribe の引数変更に注意)。
!pip install "pyannote.audio>=4.0,<5" "qwen-asr==0.0.6" fireredvad

In [ ]:
#@title ドライブ（音声置き場）マウント
#@markdown 不要なら飛ばす
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title 音声前処理 & 設定
#@markdown **音源**: 文字起こしする音声ファイルのパス
src = ""  #@param {type:"string"}
#@markdown **出力先**: 空欄なら音源と同じフォルダに同名 `.txt` を作成。指定があればそのパスに出力。フォルダ指定ならその中に音源名で作成
output_path = ""  #@param {type:"string"}
#@markdown **batch_size** / **max_tokens**（T4は既定のままでOK。OOM時はbatchを下げる）
batch_size = 8  #@param {type:"integer"}
max_tokens = 2048  #@param {type:"integer"}
#@markdown ---

#@markdown **context_label**: 語列を包む見出し。
#@markdown
#@markdown 　[issue #321](https://github.com/TypeWhisper/typewhisper-mac/issues/321) で精度UP・漏れ防止に効くとのこと。
#@markdown 試験的に日本語で適当な見出しをデフォルトで入れている。
context_label = "固有名詞・専門用語"  #@param ["固有名詞・専門用語", "Proper nouns", "Technical terms", "Vocabulary"] {allow-input: true}
#@markdown **context**: 固有名詞・専門用語を列挙（スペース/読点/カンマ区切り）
context = ""  #@param {type:"string", placeholder:"Claude Codex Gemini 競争 脱落"}
#@markdown *Tips*: Qwen3-ASRのちょっとうれしい機能Context注入
#@markdown
#@markdown 会議次第・議事レジュメのスキャン(PDF/画像)を Claude/ChatGPT に渡して「固有名詞・専門用語をスペース/読点/カンマ区切りのどれか好きな方法で列挙して」と頼み、出てきたワードをcontextに貼る。なんかこんなワード飛び交ってたなーと思ったら手動追加してもよい。
#@markdown
#@markdown なお、入れすぎると復唱(漏れ)を誘発しやすい。その場合Contextなしで再推論になるので注意

#@markdown ---
#@markdown ## FireRedVADの設定(閾値以外すべて1frame=10ms単位)
#@markdown ℹ️ まずは触らなくて問題なし
#@markdown
#@markdown **speech_threshold**: 発話判定の閾値(0～1)
speech_threshold=0.4 #@param {type:"slider", min:0, max:1, step:0.01}
#@markdown **smooth_window_size**: 前後平滑化参照フレーム数
smooth_window_size=5 #@param {type:"integer"}
#@markdown **min_speech_frame**: 発話判定の最低フレーム数
min_speech_frame=20 #@param {type:"integer"}
#@markdown **max_speech_frame**: 分割区間フレーム数
max_speech_frame=3000 #@param {type:"integer"}
#@markdown **min_silence_frame**: 区切りとみなす**無声**区間ギャップのフレーム数
min_silence_frame=20 #@param {type:"integer"}
#@markdown **merge_silence_frame**: **無声**区間を結合するフレーム数
merge_silence_frame=0 #@param {type:"integer"}
#@markdown **extend_speech_frame**: **有声**区間のオーバーラップフレーム数
extend_speech_frame=0 #@param {type:"integer"}
#@markdown **chunk_max_frame**: モデルが一度に処理するフレーム数
chunk_max_frame=30000 #@param {type:"integer"}

from fireredvad import FireRedVad, FireRedVadConfig
from huggingface_hub import snapshot_download
import os, re, librosa, numpy as np

# --- 音源パスのチェック(空 or 存在しないと librosa が意味不明なエラーで落ちるので先に弾く) ---
if not src.strip():
    raise ValueError("音源パス(src)が空です。上のフォームに音声ファイルのパスを入れてください。")
if not os.path.exists(src):
    raise FileNotFoundError(f"音源が見つかりません: {src}")

# --- issue #321 対応: 生の語列を「見出し: A、B、C。」のフレームに整形して渡す ---
#   ・裸の語列は精度を落とし漏れ(復唱)も誘発 → 見出しで包むと大幅改善＆漏れ激減
#   ・英語見出しは ", "/"." 区切り、日本語見出しは "、"/"。" 区切り
context_terms = [w for w in re.split(r"[\s、，,・･/／]+", context.strip()) if w]
if context_terms:
    sep, end = (", ", ".") if context_label.isascii() else ("、", "。")
    context_prompt = f"{context_label}: " + sep.join(context_terms) + end
else:
    context_prompt = ""

# 出力先: 指定があればそれ / 無ければ音源と同じ場所・同じ名前で .txt
# フォルダを指定した場合はその中に音源名で作る
# (下流セルが _full.txt / .srt / _Qwen_.json をこのパスから派生させる)
if output_path.strip():
    txt_output = output_path.strip()
    if os.path.isdir(txt_output):
        txt_output = os.path.join(txt_output, os.path.splitext(os.path.basename(src))[0] + ".txt")
else:
    txt_output = os.path.splitext(src)[0] + ".txt"

print("音源:", src)
print("出力:", txt_output)
print("context:", context_prompt or "(なし)")

# FireRedVADの設定
vad_config = FireRedVadConfig(
    use_gpu=False, # めちゃ軽いので不要
    smooth_window_size=smooth_window_size,
    speech_threshold=speech_threshold,
    min_speech_frame=min_speech_frame,
    max_speech_frame=max_speech_frame,
    min_silence_frame=min_silence_frame,
    merge_silence_frame=merge_silence_frame,
    extend_speech_frame=extend_speech_frame,
    chunk_max_frame=chunk_max_frame
    )

# 音声準備のためFireRedVADだけ先に読み込む
repo_dir = snapshot_download("FireRedTeam/FireRedVAD")
model_dir = f"{repo_dir}/VAD"
vad = FireRedVad.from_pretrained(model_dir, vad_config)

audio, sr = librosa.load(src, sr=16000, mono=True)

vad_audio = (audio * 32767).astype(np.int16)
result, probs = vad.detect(vad_audio)
timestamps = result["timestamps"]

print(f"検出区間数: {len(timestamps)}")
print(f"音声長: {result['dur']:.1f}秒")
for s, e in timestamps[:5]:
    print(f"  {s:6.2f} → {e:6.2f}  ({e-s:.2f}秒)")
print(f"  ... 残り{max(0, len(timestamps)-5)}区間")

In [ ]:
#@title モデル準備
#@markdown **ASRモデル** / **強制アライナ**: 通常は既定のままでOK。将来モデルが増えたらここを書き換えて差し替え。
asr_model_id     = "Qwen/Qwen3-ASR-1.7B"          #@param {type:"string"}
aligner_model_id = "Qwen/Qwen3-ForcedAligner-0.6B"  #@param {type:"string"}

import gc
import torch
from qwen_asr import Qwen3ASRModel

def _flash_ok():
    try:
        import flash_attn  # noqa: F401
        return True
    except Exception:
        return False

# --- GPU世代 + flash-attnの有無で attention実装とdtypeを自動決定 ---
# dtype : Ampere(cc>=8)以降=bfloat16 / T4等(cc<8)=float16
# attn  : cc>=8 かつ flash-attnが実際に使える時だけ flash_attention_2、それ以外は sdpa
major, minor = torch.cuda.get_device_capability(0)
compute_dtype = torch.bfloat16 if major >= 8 else torch.float16
if major >= 8 and _flash_ok():
    attn_impl = "flash_attention_2"
else:
    attn_impl = "sdpa"
    if major >= 8:
        print("⚠️ flash-attn が見つからないため sdpa で実行します（やや遅いが動作は継続）。")
print(f"GPU={torch.cuda.get_device_name(0)} (cc={major}.{minor}) -> attn={attn_impl}, dtype={compute_dtype}")
print(f"ASR={asr_model_id} / Aligner={aligner_model_id}")

if 'asr' in dir():
    del asr
    gc.collect()
    torch.cuda.empty_cache()

asr = Qwen3ASRModel.from_pretrained(
    asr_model_id,
    dtype=compute_dtype,
    device_map="cuda:0",
    attn_implementation=attn_impl,
    forced_aligner=aligner_model_id,
    forced_aligner_kwargs=dict(
        dtype=compute_dtype,
        device_map="cuda:0",
        attn_implementation=attn_impl,
    ),
    max_inference_batch_size=batch_size,
    max_new_tokens=max_tokens,
)

In [ ]:
#@title 推論
import os, numpy as np, librosa, json, re
from tqdm.auto import tqdm

# ============================================================
#@markdown ## 1.無音分割(max_clip_sec秒以下を保証)
#@markdown そのまま音声入れるとOOM必至のため、なるべく安全に音声を分割結合する処理
# ============================================================
MAX_SEC = globals().get("max_clip_sec", 30)      # 無音区間分割既定30(s)
HARD_LIMIT = MAX_SEC * sr
iv = np.array([[int(round(s*sr)), int(round(e*sr))] for s, e in timestamps])
if len(iv) == 0:
    iv = np.array([[0, len(audio)]])

# 無音区間分割の秒数を超えない程度にクリップ結合
merged = []
cs, ce = iv[0]
for s, e in iv[1:]:
    if e - cs <= HARD_LIMIT:
        ce = e
    else:
        merged.append((cs, ce)); cs, ce = s, e
merged.append((cs, ce))

# segs        = SRT用の境界(時刻・重複なし)
# clip_bounds = 推論用オーバーラップクリップ
segs, clip_bounds, n_ov = [], [], 0
for s, e in merged:
    segs.append((s, e))
    clip_bounds.append((s, e))

clips   = [(audio[a:b], sr) for a, b in clip_bounds]
offsets = [a / sr for a, b in clip_bounds]
seg_windows = [(s / sr, e / sr) for s, e in segs]
print(f"{len(clips)}クリップ / 最長: {max(e-s for s,e in segs)/sr:.1f}秒 "
      f"(FireRedVAD区間, 上限{MAX_SEC}秒, ハード再分割{n_ov}箇所)")

# 出力ファイル名作成
out_base  = os.path.splitext(txt_output)[0]
full_path = out_base + "_full.txt"
srt_path  = out_base + ".srt"
json_path = out_base + "_Qwen_.json"

# 無ければ作る
os.makedirs(os.path.dirname(out_base) or ".", exist_ok=True)
print("✅ [1/6] 無音分割 完了")

# ============================================================
#@markdown ## 2.推論
#@markdown
# ============================================================
# batch_size ごとに回して進捗バーを出す(モデル内部も同じ単位で処理するので結果は一括と同一)
_BS = globals().get("batch_size", 8) or 8
results = []
for _i in tqdm(range(0, len(clips), _BS), desc="[2/6] 推論", unit="batch"):
    _batch = clips[_i:_i + _BS]
    results.extend(asr.transcribe(
        audio=_batch,
        context=[context_prompt] * len(_batch),
        language=["Japanese"] * len(_batch),
        return_time_stamps=True,
    ))
print(f"✅ [2/6] 推論 完了（{len(results)}クリップ）")

# ============================================================
#@markdown ## 3.Context復唱時の再推論
#@markdown 最終手段的な再処理、あまり発動してほしくはない
# ============================================================
# 判定基準
#  1:見出しがそのまま出た
#  2:contextの語句を除くと虚無、つまりcontext復唱してるだけ
_ctx_words = globals().get("context_terms") or [w for w in re.split(r"[\s、，,・･/／]+", context) if w]
_label = globals().get("context_label", "")

def is_hallucination(text, ctx_words=_ctx_words, label=_label):
    t = text.strip()
    if not t:
        # そもそも無音
        return False
    if label and label in t:
        # 見出しの復唱
        return True
    if ctx_words:
        residual, hits = t, 0
        for w in sorted(ctx_words, key=len, reverse=True):
            if w in residual:
                hits += residual.count(w)
                residual = residual.replace(w, "")
        residual_clean = re.sub(r"[\s、。，,・･/／:：]+", "", residual)
        if hits >= 2 and len(residual_clean) <= 3:
            return True

    if len(t) > 150 and (t.count("。") + t.count("、")) < 3:
        return True
    return False

fixed = []
for i, r in enumerate(tqdm(results, desc="[3/6] 漏れチェック", unit="clip")):
    if is_hallucination(r.text):
        r_fix = asr.transcribe(
            audio=[clips[i]],
            language=["Japanese"],
            return_time_stamps=True,
        )
        results[i] = r_fix[0]
        fixed.append(i)

if fixed:
    print(f"漏れ検出→contextなしで再処理したクリップ: {fixed}")
    for i in fixed:
        print(f"  [{i}] 修正後: {results[i].text[:50]}")
else:
    print("漏れなし(全クリップ正常)")
print("✅ [3/6] 漏れチェック 完了")

# ============================================================
#@markdown ## 4.句読点付きの書き出し
#@markdown 元データ
# ============================================================
def hms(t):
    return f"{int(t//3600):02}:{int(t%3600//60):02}:{int(t%60):02}"

with open(full_path, 'w', encoding='utf-8') as f:
    for off, r in zip(offsets, results):
        f.write(f"[{hms(off)}] {r.text}\n")
print(f"✅ [4/6] full.txt 書き出し 完了（{len(results)}セグメント）")

# ============================================================
#@markdown ## 5.SRT書き出し
#@markdown これさえあれば議事録はなんとかなる
# ============================================================
def srt_t(t):
    return f"{hms(t)},{int((t % 1) * 1000):03}"

# 日本語対策らしい
# 触らないほうが良い
MAX_DUR   = 6.0   # 1字幕の最大長(秒)
MAX_CHARS = 30    # 1字幕の最大文字数
GAP       = 0.8   # これ以上の無音で改行(秒)

def clip_words(off, win, r):
    ws, we = win
    return [ts for ts in (r.time_stamps or [])
            if ws <= off + (ts.start_time + ts.end_time) / 2 < we]

def join_tokens(toks):
    out = ""
    for t in toks:
        if out and out[-1].isascii() and out[-1].isalnum() and t and t[0].isascii() and t[0].isalnum():
            out += " "
        out += t
    return out

idx = 1
with open(srt_path, 'w', encoding='utf-8') as f:
    for off, win, r in zip(offsets, seg_windows, results):
        buf = []
        for ts in clip_words(off, win, r) + [None]:
            if buf and (ts is None
                        or ts.start_time - buf[-1].end_time >= GAP
                        or ts.end_time - buf[0].start_time > MAX_DUR
                        or sum(len(b.text) for b in buf) >= MAX_CHARS):
                f.write(f"{idx}\n{srt_t(off + buf[0].start_time)} --> {srt_t(off + buf[-1].end_time)}\n"
                        + join_tokens([b.text for b in buf]) + "\n\n")
                idx += 1
                buf = []
            if ts is not None:
                buf.append(ts)
print(f"✅ [5/6] SRT 書き出し 完了（{idx - 1}エントリ）")

# ============================================================
#@markdown ## 6.JSON書き出し
#@markdown Pyannoteの話者分離用
# ============================================================
json_segments = []
for off, win, r in zip(offsets, seg_windows, results):
    words = clip_words(off, win, r)
    json_segments.append({
        "start": win[0],
        "end": (off + words[-1].end_time) if words else win[0],
        "text": r.text,
        "words": [
            {"word": ts.text, "start": off + ts.start_time, "end": off + ts.end_time}
            for ts in words
        ],
    })

with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(json_segments, f, ensure_ascii=False, indent=2)
print(f"✅ [6/6] JSON 書き出し 完了（{len(json_segments)}セグメント）")

print(f"\n保存完了:")
print(f"  {os.path.basename(full_path)} / {os.path.basename(srt_path)} / {os.path.basename(json_path)} ({len(json_segments)}セグメント)")

# 話者分離編

上の文字起こし（[6/6] JSON書き出しまで）が終わった後に続けて実行する追加パート

## 事前準備（初回のみ）
1. Hugging Face Read Token：https://huggingface.co/settings/tokens
2. gatedなので同意
   - 4.x：https://huggingface.co/pyannote/speaker-diarization-community-1
   - 3.1：https://huggingface.co/pyannote/speaker-diarization-3.1 と https://huggingface.co/pyannote/segmentation-3.0
3. ColabのSecretsに`HF_TOKEN`で登録

> - OOM になりそうなら設定セルの `free_asr_vram` をON。

In [ ]:
#@title 話者分離設定
#@markdown **話者分離モデル**: 既定は community-1
diar_model_id = "pyannote/speaker-diarization-community-1"  #@param ["pyannote/speaker-diarization-community-1", "pyannote/speaker-diarization-3.1"] {allow-input: true}
#@markdown **num_speakers**: 会議の発言者数、正しく設定すると精度が上がる(0=自動推定)
num_speakers = 0  #@param {type:"integer"}
#@markdown **free_asr_vram**: ASR分のVRAM解放。OOMが出たらTrueに
free_asr_vram = False  #@param {type:"boolean"}

import gc, torch

if free_asr_vram and 'asr' in dir():
    del asr
    gc.collect()
    torch.cuda.empty_cache()
    print("ASRモデルをVRAMから解放した")

# HF_TOKEN取得 → なければCLI入力
hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    pass
if not hf_token:
    from getpass import getpass
    hf_token = getpass("HF_TOKEN (hf_...): ").strip()

from pyannote.audio import Pipeline

try:
    diar_pipe = Pipeline.from_pretrained(diar_model_id, token=hf_token)
except TypeError:
    # 古いpyannoteは引数名が use_auth_token
    diar_pipe = Pipeline.from_pretrained(diar_model_id, use_auth_token=hf_token)

if diar_pipe is None:
    raise RuntimeError(
        f"パイプラインを取得できませんでした: {diar_model_id}\n"
        "Gatedモデルなので規約の同意とトークン必須")

diar_pipe.to(torch.device("cuda:0"))
print(f"✅ {diar_model_id} ロード完了")

In [ ]:
#@title 推論
#@markdown **qwen_json**: 通常は空欄でよい(推論セルの結果を使う)。明示的に支持できるだけ。
qwen_json = ""  #@param {type:"string"}

import os, glob, json, bisect, torch, numpy as np

# --- 入力の解決: 再起動後でも qwen_json さえあれば動くように復元 ---
if qwen_json.strip():
    json_path = qwen_json.strip()
if 'json_path' not in dir() or not os.path.exists(json_path):
    raise RuntimeError("_Qwen_.json が見つかりません。上の推論セルまで実行するか、qwen_json にパスを指定してください。")
if 'out_base' not in dir():
    out_base = json_path[:-len("_Qwen_.json")] if json_path.endswith("_Qwen_.json") else os.path.splitext(json_path)[0]
if 'audio' not in dir():
    if 'src' not in dir():
        # 音源はout_baseと同名の音声ファイルを探す(出力先を変えている場合は音声前処理セルでsrcを定義)
        _c = [p for p in glob.glob(glob.escape(out_base) + ".*")
              if p.lower().endswith((".m4a", ".mp3", ".wav", ".flac", ".ogg", ".aac"))]
        if not _c:
            raise RuntimeError(f"音源を自動検出できません: {out_base}.* (音声前処理 & 設定セルを実行してsrcを定義してください)")
        src = _c[0]
        print("音源を自動検出:", src)
    import librosa
    audio, sr = librosa.load(src, sr=16000, mono=True)

# ============================================================
#@markdown ## 1. 話者分離
#@markdown ASRでの前処理後の音声をそのまま渡す
# ============================================================
wav = torch.from_numpy(np.asarray(audio, dtype=np.float32)).unsqueeze(0)
diar_kwargs = {"num_speakers": num_speakers} if num_speakers and num_speakers > 0 else {}
diar_out = diar_pipe({"waveform": wav, "sample_rate": sr}, **diar_kwargs)
# pyannote 4.xは属性、3.1系はAnnotationが返る
ann = getattr(diar_out, "speaker_diarization", diar_out)

turns = sorted((t.start, t.end, spk) for t, _, spk in ann.itertracks(yield_label=True))
speakers = sorted({spk for *_, spk in turns})
print(f"✅ [1/3] 話者分離 完了（話者{len(speakers)}人 / ターン{len(turns)}個）")

# ============================================================
#@markdown ## 2. タイムスタンプと照合
#@markdown 各単語の中点時刻に話者を割り当て
# ============================================================
with open(json_path, encoding="utf-8") as f:
    qwen_segments = json.load(f)

_starts = [s for s, _, _ in turns]

def _speaker_at(mid):
    i = bisect.bisect_right(_starts, mid) - 1
    cand = [j for j in (i, i + 1) if 0 <= j < len(turns)]
    if not cand:
        return "SPEAKER_??"
    for j in cand:
        s, e, spk = turns[j]
        if s <= mid <= e:
            return spk
    j = min(cand, key=lambda j: min(abs(mid - turns[j][0]), abs(mid - turns[j][1])))
    return turns[j][2]

spk_words = []
for seg in qwen_segments:
    for w in seg["words"]:
        mid = (w["start"] + w["end"]) / 2
        spk_words.append({**w, "speaker": _speaker_at(mid)})
spk_words.sort(key=lambda w: w["start"])
print(f"✅ [2/3] 話者割り当て 完了（{len(spk_words)}単語）")

# ============================================================
#@markdown ## 3. 話者付き書き出し
#@markdown `_speakers.txt`(議事録用) と `_speakers.srt`(話者付き字幕) と `_QwenPyannote_.json`(単語レベルのマージ結果)
# ============================================================
def hms(t):
    return f"{int(t//3600):02}:{int(t%3600//60):02}:{int(t%60):02}"

def srt_t(t):
    return f"{hms(t)},{int((t % 1) * 1000):03}"

def join_tokens(toks):
    out = ""
    for t in toks:
        if out and out[-1].isascii() and out[-1].isalnum() and t and t[0].isascii() and t[0].isalnum():
            out += " "
        out += t
    return out

# 改行処理: 話者交代 or 同一話者でもUTT_GAP秒以上空いたら行を分ける
UTT_GAP = 10.0
utterances = []
for w in spk_words:
    if (utterances
            and utterances[-1]["speaker"] == w["speaker"]
            and w["start"] - utterances[-1]["end"] < UTT_GAP):
        utterances[-1]["words"].append(w)
        utterances[-1]["end"] = w["end"]
    else:
        utterances.append({"speaker": w["speaker"], "start": w["start"], "end": w["end"], "words": [w]})
for u in utterances:
    u["text"] = join_tokens([w["word"] for w in u["words"]])

spk_txt_path  = out_base + "_speakers.txt"
spk_srt_path  = out_base + "_speakers.srt"
spk_json_path = out_base + "_QwenPyannote_.json"

with open(spk_txt_path, "w", encoding="utf-8") as f:
    for u in utterances:
        f.write(f"[{hms(u['start'])}] {u['speaker']}: {u['text']}\n")

# 話者付きSRT: 分割基準はASR側のSRTセルと同じ値 + 話者交代でも必ず区切る
# (話者ラベル分はMAX_CHARSに数えない)
MAX_DUR   = 6.0   # 1字幕の最大長(秒)
MAX_CHARS = 30    # 1字幕の最大文字数
GAP       = 0.8   # これ以上の無音で改行(秒)

idx, buf = 1, []
with open(spk_srt_path, "w", encoding="utf-8") as f:
    for w in spk_words + [None]:
        if buf and (w is None
                    or w["speaker"] != buf[0]["speaker"]
                    or w["start"] - buf[-1]["end"] >= GAP
                    or w["end"] - buf[0]["start"] > MAX_DUR
                    or sum(len(b["word"]) for b in buf) >= MAX_CHARS):
            f.write(f"{idx}\n{srt_t(buf[0]['start'])} --> {srt_t(buf[-1]['end'])}\n"
                    + f"{buf[0]['speaker']}: " + join_tokens([b["word"] for b in buf]) + "\n\n")
            idx += 1
            buf = []
        if w is not None:
            buf.append(w)

with open(spk_json_path, "w", encoding="utf-8") as f:
    json.dump(utterances, f, ensure_ascii=False, indent=2)

print(f"✅ [3/3] 書き出し 完了（{len(utterances)}発話 / SRT {idx - 1}エントリ）")
print("\n保存完了:")
print(f"  {os.path.basename(spk_txt_path)} / {os.path.basename(spk_srt_path)} / {os.path.basename(spk_json_path)}")